# Imports

In [1]:
from WS_Mdl.core import *

In [2]:
import datetime as DT
import pandas as pd

In [3]:
from WS_Mdl.imod.sfr.info import SFR_PkgD_to_DF

# Prep

In [4]:
MdlN = 'NBr70'
MdlN_B = 'NBr68'

In [5]:
M = Mdl_N(MdlN)
MB = Mdl_N(MdlN_B)

# Load + process DF

In [20]:
DF = SFR_PkgD_to_DF(MB.MdlN)

🟡 - Coordinates (X, Y columns) not found in PACKAGEDATA. Calculating coordinates from INI file info.


In [21]:
DF

,rno,k,i,j,Cond,rlen,rwid,rgrd,rtp,rbth,rhk,man,ncon,ustrf,ndv,line_id,x,y
0,1,7,282,393,1.97675,4.667041,0.5,0.0001,18.799999,0.1,0.084711,0.037,1,1,0,297,122912.5,389162.5
1,2,7,281,393,6.917482,14.09273,0.5,0.0001,18.799999,0.1,0.098171,0.037,2,1,0,297,122912.5,389187.5
2,3,7,281,392,17.316629,26.508957,0.5,0.0001,18.799999,0.1,0.130647,0.037,2,1,0,297,122887.5,389187.5
3,4,7,281,391,11.49135,27.918734,0.5,0.0001,18.799999,0.1,0.08232,0.037,2,1,0,297,122862.5,389187.5
4,5,7,281,390,7.758311,18.279442,0.5,0.0001,18.799999,0.1,0.084886,0.037,2,1,0,297,122837.5,389187.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3657,3658,5,24,17,38.177928,22.245678,4.88,0.0001,-0.1,0.1,0.035168,0.037,2,1,0,4678,113512.5,395612.5
3658,3659,5,24,16,42.734611,23.648016,4.88,0.0001,-0.1,0.1,0.037031,0.037,2,1,0,4678,113487.5,395612.5
3659,3660,5,23,16,9.524576,5.949796,4.88,0.117651,-0.1,0.1,0.032804,0.037,2,1,0,4678,113487.5,395637.5
3660,3661,5,23,15,44.393411,14.616602,5.0,0.0001,-0.8,0.1,0.060744,0.037,2,1,0,4667,113462.5,395637.5


# Conert to DA

In [13]:
DA = DF.set_index(['y', 'x'])[['rtp']].to_xarray()

In [ ]:
DA['rtp']

# Save SFR+RIV BOT .IDF
We need to do this cause the SFR only has values inside the catchment.

## Load RIV

In [18]:
from WS_Mdl.xr.spatial import clip_Mdl_area

In [19]:
import xarray as xra

In [22]:
import imod

In [23]:
RIV_BOT = clip_Mdl_area(imod.idf.open(M.Pa.Mdl / r"In\RIV\NBr49\RIV_Bot_main_NBr49.IDF"), M.MdlN)

🟡 - Reversed y-axis of DataArray to match model area orientation.


# Align Coordinates, Join, Save

In [ ]:
RIV_BOT, RIV_BOT = xra.align(RIV_BOT, DA['rtp'], join='outer')

TypeError: Cannot interpret 'Float64Dtype()' as a data type

In [ ]:
imod.idf.save(M.Pa.In / f'RIV/{MdlN}/RIV_Stg_main_winter_{M.MdlN}.IDF', xra.where(SFR_Stg_winter.notnull(), SFR_Stg_winter, RIV_Stg_winter))
imod.idf.save(M.Pa.In / f'RIV/{MdlN}/RIV_Stg_main_summer_{M.MdlN}.IDF', xra.where(SFR_Stg_summer.notnull(), SFR_Stg_summer, RIV_Stg_summer))